# craft-support

> Bug-to-review-app support and experimentation workflow for `solve-it-crafts`.

`craft-support` turns browser bug captures from Solveit dialogs into sanitized Google Drive ticket bundles, interactive terminal triage sessions, GitHub pull requests, and isolated Solveit review apps — built end-to-end on the Answer.AI stack.

## Architecture

```
Browser Capture Widget (19_bug_capture.ipynb)
        │  POST /api/tickets (CORS + Private Network Access)
        ▼
FastHTML Ingestion Server (03_server.ipynb)
        │  sanitize_dom() via fast5ever (01_sanitize.ipynb)
        ▼
Google Drive Ticket Store (02_drive.ipynb)  ◄──►  Textual Triage TUI (06_tui.ipynb)
  solveitcrafts/<ticket_id>/                        & craft-support CLI (07_cli.ipynb)
    ticket.json | dom.html | screenshot.png
        │
        ▼  ReviewOrchestrator (05_orchestrator.ipynb)
  ┌─────┴──────────────────────┬──────────────────────────────────┐
  │ fastgit (non-destructive   │ ghapi (open/reuse PR, comment    │ fastcdp (04_solveit_cdp.ipynb)
  │ commit-tree + update-ref)  │ review-app link, merge/close)    │ create/upload/delete Solveit app
  └────────────────────────────┴──────────────────────────────────┴──────────────────────────────────┘
```

### Answer.AI Stack

- **`nbdev` v3**: Literate source notebooks (`nbs/*.ipynb`) that export `craft_support/*.py`, run inline tests (`nbdev-test`), and build docs (`nbdev-docs`, `nbdev-readme`).
- **`AnswerDotAI/fastship`**: Package scaffolding, version bumps (`ship-bump`), and PyPI/GitHub releases (`ship-pypi`, `ship-gh`).
- **`AnswerDotAI/fastgws`**: Google Workspace OAuth (via `gclientid`) and resilient async Drive API transport.
- **`AnswerDotAI/solvedrive`**: High-level async Google Drive client (`GDrive`, `Folder`, `File`) for ticket bundles (`solveitcrafts/<ticket_id>/`).
- **`AnswerDotAI/fast5ever`**: HTML DOM parser and sanitizer that strips scripts, event handlers, and sensitive form/auth values before anything is persisted.
- **`AnswerDotAI/fastgit`**: Local git branch creation, stacking, pushing, and cleanup without disturbing uncommitted working-tree edits.
- **`fastai/ghapi`**: GitHub pull request creation, linking Drive tickets, commenting with review-app URLs, merging, and closing.
- **`fastcdp`**: Provisioning, inspecting, and tearing down isolated Solveit review-app instances via `solve.it.com/dashboard`.

## Notebook Layout

- `nbs/00_models.ipynb` → `craft_support/models.py` — `Ticket`, `CaptureMeta`, `TriageNote`, `DriveRefs`, `ReviewManifest`, lifecycle statuses.
- `nbs/01_sanitize.ipynb` → `craft_support/sanitize.py` — `fast5ever` HTML DOM sanitizer (`sanitize_dom`, `SanitizeResult`).
- `nbs/02_drive.ipynb` → `craft_support/drive.py` — `TicketStore`, `MemoryTicketStore`, `LocalTicketStore`, `DriveTicketStore`.
- `nbs/03_server.ipynb` → `craft_support/server.py` — FastHTML ingestion server with Private Network Access CORS headers.
- `nbs/04_solveit_cdp.ipynb` → `craft_support/solveit_cdp.py` — Review-app bundle zip builder, `MemoryReviewAppProvisioner`, and `FastCDPReviewAppProvisioner`.
- `nbs/05_orchestrator.ipynb` → `craft_support/orchestrator.py` — `FastGitAdapter`, `GhApiAdapter`, and idempotent `ReviewOrchestrator`.
- `nbs/06_tui.ipynb` → `craft_support/tui.py` — Textual `TriageApp` and modals (`NoteModal`, `StatusModal`, `AssignModal`).
- `nbs/07_cli.ipynb` → `craft_support/cli.py` — `craft-support` CLI entry point.

## Quickstart

### Install

```bash
cd support
uv venv --python 3.13 .venv
uv pip install --python .venv/bin/python -e ".[dev]"
```

### Local Offline Smoke Run

You can run the full ticket creation, sanitization, and triage flow offline without Google OAuth or GitHub credentials:

In [ ]:
import asyncio
from craft_support import MemoryTicketStore

In [ ]:
async def _smoke():
    store = MemoryTicketStore()
    ticket, san = await store.create_ticket(
        title="Sidebar toggle clips on narrow viewports",
        description="Mod+B hides the tree but leaves a 12px gutter.",
        raw_dom_html='<div onclick="steal()"><script>alert(1)</script><input type="password" value="secret123"><p>Visible bug text</p></div>')
    dom = await store.get_dom(ticket.id)
    assert "<script" not in dom.lower() and "secret123" not in dom and "Visible bug text" in dom
    return f"Created {ticket.id} [{ticket.status}] (stripped={san.stripped_tags}, redacted={san.redacted_fields})"

await _smoke()

### Live Google Workspace OAuth, GitHub & Solveit

1. Configure Google Workspace OAuth once for Drive storage:

```bash
gclientid
gclientid-auth you@example.com --preset google-apps
```

2. Ensure GitHub auth is available via `GITHUB_TOKEN`, `GH_TOKEN`, or `gh auth login`, and Solveit cookies are saved via `uv run craft-cdp auth` from the repository root.

3. Run the ingestion server, TUI, or CLI commands:

```bash
# Start the bug-capture ingestion endpoint (Google Drive backed)
.venv/bin/craft-support --account you@example.com serve --port 8791

# Or test offline with the local filesystem mirror
.venv/bin/craft-support --store local serve --port 8791

# Launch the interactive terminal triage UI
.venv/bin/craft-support --account you@example.com tui

# Scriptable CLI equivalents
.venv/bin/craft-support list --status new
.venv/bin/craft-support show TKT-20261123-ABC123
.venv/bin/craft-support note TKT-20261123-ABC123 "Reproduced on mobile"
.venv/bin/craft-support assign TKT-20261123-ABC123 --base-branch main
.venv/bin/craft-support merge TKT-20261123-ABC123
.venv/bin/craft-support close TKT-20261123-ABC123
.venv/bin/craft-support teardown review-support-tkt-20261123-abc123
```

### In-App Capture Widget

The toggleable help icon and bug-report modal live in `nbs/craft/19_bug_capture.ipynb`. Compile with `uv run craft-cdp build` from the repository root. Toggle the icon from the Solveit command launcher (`Mod+Shift+P` → *Toggle bug-report help icon*) or `window.craftBugCapture.setEnabled(true)`.

### Development, Testing & Docs

```bash
.venv/bin/nbdev-export   # export nbs/*.ipynb -> craft_support/*.py
.venv/bin/nbdev-test     # run inline notebook tests (skips #| live / #| notest)
.venv/bin/nbdev-readme   # regenerate README.md from nbs/index.ipynb
.venv/bin/nbdev-clean    # strip dirty notebook metadata
.venv/bin/nbdev-docs     # build Quarto documentation into _docs/
.venv/bin/pytest -v      # run end-to-end pytest suite
```